# 08 · Final models, export and test

Everything here goes through the same trainer as the search, with `[Final]` of the configuration:

- **the winner** of the search trained again, 40,000 batches, seeds 0, 1 and 2, in FP16 with distillation and BatchNorm folded for the last 20% of the steps;
- **baselines**: U-Nets of segmentation_models_pytorch with the EfficientNet-B0 and MobileNetV2 encoders, off the shelf and with the decoder slimmed by hand to fit 5 MB, trained the same way;
- **ablations** of the winner: without distillation (`no_kd`), and trained in FP32 with BatchNorm frozen instead of folded, converted to FP16 only afterwards (`fp32`), as a network is usually brought to the Myriad.

All are scored on the search and validation sets. The deployed model is the winner's seed with the best validation score in FP16; it is exported as the files ArgOS takes and checked. The test split is read once, at the end, by `final.test`, which keeps its result and never computes it again.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, final, report, runner
from pynas.hardware import myriad

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
CONFIG = Path("configs/burnscar.ini")
cfg = config.load(CONFIG)
BASE = Path(cfg.get("Run", "out"))
SEARCH, FINAL, TEACHER_Q = BASE / "search", BASE / "final", BASE / "teacher" / "runs" / "queue.json"
GPUS = list(range(torch.cuda.device_count())) or [0]
w = final.winner(SEARCH)
FQ = FINAL / "runs" / "queue.json"
DEP = Path("models/onboard/burnscar_pynas/deploy.json")
dep = json.load(open(DEP)) if DEP.exists() else None
print(f"PyNAS {pynas.__version__}; winner {w['spec']} (search score {w['score']:.4f}, {w.get('latency_ms')} ms)")

In [ ]:
FQ = FINAL / "runs" / "queue.json"
dep = json.load(open("models/onboard/burnscar_pynas/deploy.json"))

In [ ]:
# [2] the final runs, in the background (running the cell again carries on)
FQ = final.queue(cfg, SEARCH, FINAL)
print(f"{len(runner.read_queue(FQ)['runs'])} runs")
runner.launch(FQ, gpus=GPUS)

In [ ]:
# [3] progress every 15 minutes (interrupting stops the watching, not the trainings)
runner.wait(FQ, every=900)

In [ ]:
# [4] fresh-burn IoU in FP16 on the search and validation sets: mean and standard deviation over the seeds
report.final_table(FQ, cfg).round(3)

## Export

The winner's seed with the best mean validation IoU in FP16, as `models/onboard/burnscar_pynas/`: FP16 IR of the OpenVINO 2022.3.1 Model Optimizer (`.xml`, `.bin`, `.mapping`), checked by compiling it for MYRIAD; its IR run on the OpenVINO CPU plugin against PyTorch on validation windows (a check of the conversion); the share of pixels where FP16 gives the class FP32 gives.

In [ ]:
# [5] the deployed model
dep = final.deploy(FQ, cfg, "models/onboard", "burnscar_pynas")
print(json.dumps({k: v for k, v in dep.items() if k not in ("layers", "ir_vs_pytorch")}, indent=1, default=str))
if dep.get("ir_vs_pytorch"):
    display(pd.DataFrame(dep["ir_vs_pytorch"]).set_index("set"))

In [ ]:
# [6] the hardware numbers of the winner and the baselines (one seed each): parameters of the exported network, IR size, compiled for MYRIAD, latency
from pynas.hardware.latency import latency_ms
from pynas.space.network import n_params_used
res = final.results(FQ)
one = res[res.role.isin(["final", "baseline"])].drop_duplicates("label")
nets = {f"m{i}": runner.load_result(FINAL / "runs" / r.run) for i, r in enumerate(one.itertuples())}
shape = tuple(json.load(open(FINAL / "runs" / one.run.iloc[0] / "run.json"))["input_shape"])
hw = myriad.check(nets, FINAL / "hardware", shape, cfg) if myriad.status(cfg)["model_optimizer"] else {}
lat = latency_ms(nets, shape, cfg) if myriad.status(cfg)["nnmeter"] else {}
pd.DataFrame([dict(model=r.label, params_M=n_params_used(nets[k], shape) / 1e6, ir_mb=hw.get(k, {}).get("ir_mb"),
                   compiled=hw.get(k, {}).get("compiled"), latency_ms=lat.get(k))
              for k, r in zip(nets, one.itertuples())]).set_index("model").round(3)

## Test

Fresh-burn IoU and burn IoU (fresh + old) on every test set: the onboard models in FP16 (as they run on the Myriad) and in FP32, the teacher in FP32. Mean and standard deviation over the seeds; the deployed seed is one of the winner's. Run once; afterwards the cell reads the stored result.

In [ ]:
# [7] the test split, read once
scores, per_gsd = final.test(FQ, cfg, teacher_queue=TEACHER_Q)
deployed = scores[scores.run == dep["run"]].assign(label="deployed (" + scores.label + f", seed {dep['seed']})")
display(report.test_table(pd.concat([scores, deployed]), cfg).round(3))

In [ ]:
report.final_table(FQ, cfg).round(4).to_csv("final_table.csv")
report.test_table(pd.concat([scores, deployed]), cfg).round(4).to_csv("test_table.csv")
per_gsd.to_csv("test_per_gsd.csv", index=False)

In [ ]:
# [8] fresh-burn IoU per resolution on the test sets (mean over seeds)
per_gsd.groupby(["label", "precision", "set", "gsd"], sort=False).iou.mean().unstack(["set", "gsd"]).round(3)

## Test results by resolution and by scene

Fresh-burn IoU on each test set over all its windows and at each resolution it is scored at, mean and standard deviation over the seeds (the onboard models in FP16, the teacher in FP32). HEO scenes are scored at 2.5, 5 and 10 m: the HEO image resampled to windows of that ground size, as the training windows are drawn between 1 and 10 m. PhiSat-2 is scored at 4.75 m, its own resolution, and at 10 m; Sentinel-2 at its own 10 m.

Then the test scenes themselves, every window put back in its place: the image (NIR, red, green), the labels, and for the deployed PyNAS model, the two slimmed U-Nets and the teacher (mean of its two seeds) the fresh burn found (yellow), the false alarms (magenta) and the misses (cyan), with the IoU of that scene at that resolution. For PhiSat-2, the three scenes with the most fresh burn among its sampled test windows; the gaps are windows outside the sample.

In [ ]:
# [9] fresh-burn IoU on every test set, over all windows and per resolution
ORDER = ["winner", "smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8", "smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8",
         "smp:Unet:efficientnet-b0", "winner, no_kd", "winner, fp32", "teacher"]
report.plot_test_iou(per_gsd, labels=ORDER)
plt.tight_layout(); plt.savefig(FINAL / "test_iou_by_gsd.png", dpi=200); plt.show()

In [ ]:
# [10] the test scenes
from pynas import teacher
from pynas.train import data as D
spec = D.DataSpec.from_config(cfg)
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
dep = json.load(open("models/onboard/burnscar_pynas/deploy.json"))
res = final.results(FQ)
seed0 = lambda label: runner.load_result(FINAL / "runs" / res[(res.label == label) & (res.seed == 0)].run.iloc[0], dev)
models = {f"PyNAS (deployed, seed {dep['seed']})": (runner.load_result(FINAL / "runs" / dep["run"], dev), "fp16"),
          "U-Net EfficientNet-B0, slim": (seed0("smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8"), "fp16"),
          "U-Net MobileNetV2, slim": (seed0("smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8"), "fp16"),
          "teacher (mean of seeds)": (final.Mean(teacher.models(TEACHER_Q, dev)), "fp32")}
for name in spec.test:
    s = D.open_set(spec, name, teacher=False)
    fig = report.plot_scenes(s, report.predict_set(models, spec, s, dev), spec, name, max_scenes=3)
    fig.savefig(FINAL / f"test_scenes_{name}.png", dpi=200); plt.show()

Next: the ArgOS test suite on `models/onboard/burnscar_pynas/` (its `.xml`, `.bin` and `.mapping` go to `argos_sdk/models/`), and the model on the Myriad X itself.